# Formula 1 World Championship Analysis
## Machine Learning
* **Goal:** Predict the probability of a Formula 1 driver winning a race based on historical driver performance, constructor performance and qualifying position.

In [42]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    confusion_matrix, 
    classification_report
)
from sklearn.dummy import DummyClassifier

In [26]:
# Safe data loading with try/except
file_path = ('../data/formula_dataset_ml.csv')

try:
    df = pd.read_csv(file_path)
    print('File loaded successfully')
except FileNotFoundError:
    print(f'File not Found - Check {file_path}')
except Exception as e:
    print(f'Error occurred - Check {e}')

File loaded successfully


In [27]:
# Inspection of Dataset

print(df.shape)
print(df.head())
print(df.columns)

(26759, 15)
   grid  pole_position  year  previous_races  previous_wins  \
0     1              1  1950               0              0   
1     2              0  1950               0              0   
2     4              0  1950               0              0   
3     6              0  1950               0              0   
4     9              0  1950               0              0   

   previous_win_rate  previous_points  previous_avg_position  \
0                0.0              0.0                    NaN   
1                0.0              0.0                    NaN   
2                0.0              0.0                    NaN   
3                0.0              0.0                    NaN   
4                0.0              0.0                    NaN   

   previous_podiums  previous_podium_rate  previous_constructor_wins  \
0                 0                   0.0                          0   
1                 0                   0.0                          0   
2       

In [28]:
# Define features and target
X = df.drop('win', axis=1)
y = df['win']

print('Features shape:', X.shape)
print('Target shape:', y.shape)

Features shape: (26759, 14)
Target shape: (26759,)


In [29]:
# Check target distribution
print(y.value_counts())
print()
print(y.value_counts(normalize=True))

win
0    25631
1     1128
Name: count, dtype: int64

win
0    0.957846
1    0.042154
Name: proportion, dtype: float64


In [30]:
# Checking for the missing values
print(X.isna().sum())

grid                               0
pole_position                      0
year                               0
previous_races                     0
previous_wins                      0
previous_win_rate                  0
previous_points                    0
previous_avg_position            861
previous_podiums                   0
previous_podium_rate               0
previous_constructor_wins          0
previous_constructor_points        0
previous_constructor_races         0
previous_constructor_win_rate      0
dtype: int64


In [31]:
# Check available seasons
print(df['year'].min())
print(df['year'].max())

1950
2024


In [32]:
# Define ML features
ml_features = [
    'grid',
    'pole_position',
    'previous_races',
    'previous_wins',
    'previous_win_rate',
    'previous_points',
    'previous_avg_position',
    'previous_podiums',
    'previous_podium_rate',
    'previous_constructor_wins',
    'previous_constructor_points',
    'previous_constructor_races',
    'previous_constructor_win_rate'
]

target = 'win'

print('Number of features:', len(ml_features))
print('Target:', target)

Number of features: 13
Target: win


In [33]:
# Split data chronologically
train_df = df[df['year']<= 2019].copy()
test_df = df[df['year'] >= 2020].copy()

print('Training set:', train_df.shape)
print('Test set:', test_df.shape)

Training set: (24620, 15)
Test set: (2139, 15)


In [34]:
# Check of the right splitting

# Trainset
print(
    'Training years:',
    train_df['year'].min(), 
    '-',
    train_df['year'].max()
)

# Testset
print(
    'Test years:', 
    test_df['year'].min(), 
    '-',
    test_df['year'].max()
)

Training years: 1950 - 2019
Test years: 2020 - 2024


In [35]:
# Define training and test features and target

X_train = train_df[ml_features]
y_train = train_df[target]

X_test = test_df[ml_features]
y_test = test_df[target]

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (24620, 13)
y_train: (24620,)
X_test: (2139, 13)
y_test: (2139,)


In [43]:
# Create imputer

imputer = SimpleImputer(strategy='median')

# Fit on training data and transform training data
X_train_imputer = imputer.fit_transform(X_train)
X_test_imputer = imputer.transform(X_test)

In [41]:
# Check the sum of missing values
print('Missing values in X_train:', np.isnan(X_train_imputer).sum())
print('Missing valeus in X_test:', np.isnan(X_test_imputer).sum())

Missing values in X_train: 0
Missing valeus in X_test: 0


In [44]:
# Create baseline model

baseline = DummyClassifier(strategy='most_frequent')

baseline.fit(X_train_imputer, y_train)
baseline_prediction = baseline.predict(X_test_imputer)

In [46]:
# Check Accuracy Score

baseline_accuracy = accuracy_score(y_test, baseline_prediction)
print(f'Baseline Accuracy Score: {baseline_accuracy*100:.2f}%')


Baseline Accuracy Score: 95.00%
